# WorldSplat-Prune — front-end smoke test (caption → GroundingDINO → SAM2 → instance maps)

Exercises the **prompt-driven labelling front-end** (§2.2) end to end on one scene
and stops at the **instance-label map** output — the post-reconstruction pruning
operator is not involved.

| stage | module | output |
|---|---|---|
| 1 | `frontend.caption` | seed phrase list (`phrases_seed.txt`) |
| 2 | `frontend.grounding` | per-frame boxes (`detections.json`) |
| 3 | `frontend.sam2_box` | per-class instance maps (`semantic_instance_<cls>/`) |
| 4 | `prep.fuse_instances` | fused per-class ids + composite maps (`semantic_composite/`) |

Every stage is skippable and its intermediate artifact is kept on disk, so a
failure at one stage doesn't cost the earlier work. Runs on a Colab GPU (a T4 is
enough with the `-tiny` detector and a subsampled `STRIDE`).

> **Does this work at the current state?** Yes — the front-end modules exist
> (`worldsplat_prune/frontend/`, `worldsplat_prune/datasets/composite.py`,
> `worldsplat_prune/prep/fuse_instances.fuse_instances_per_class`). Their **pure**
> parts are unit-tested; the **model** paths (`torch` / `transformers` / `sam2`)
> are exercised for the first time here, so treat this as the smoke test.


## 1. Mount Google Drive

Skip this cell if you run locally / already have the dataset on disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get this repo + front-end deps

Colab already ships PyTorch. We add `transformers` + `timm` (GroundingDINO and the
captioner) and install SAM2 from source. The CUDA rasterizer submodules are **not**
needed for labelling, but we init them so the same checkout can train afterwards.


In [ ]:
import os

REPO_URL = "https://github.com/satyaba/worldsplat-prune.git"
REPO_DIR = "/content/worldsplat-prune"
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn

!pip install -q plyfile opencv-python joblib
!pip install -q transformers timm
!pip install -q "git+https://github.com/facebookresearch/sam2.git"

print("cwd:", os.getcwd())

## 3. GPU check

GroundingDINO and SAM2 both run on GPU; CPU works but is slow. The models are small
(`grounding-dino-tiny`, `sam2.1_hiera_large`) so a T4 suffices.


In [ ]:
!nvidia-smi

import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configure paths


In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_NAME = "office_0/Sequence_1"          # or e.g. "mipnerf360/garden"

os.environ["HOME_DIR"] = HOME_DIR
os.environ["DATASET_ROOT"] = str(Path(HOME_DIR) / "Dataset")
DATASET_DIR = Path(os.environ["DATASET_ROOT"]) / DATASET_NAME

n_images = len([p for p in (DATASET_DIR / "images").glob("*") if p.is_file()]) \
    if (DATASET_DIR / "images").is_dir() else 0
print("DATASET_DIR :", DATASET_DIR)
print("images      :", n_images)
print("colmap      :", (DATASET_DIR / "sparse" / "0" / "cameras.bin").exists()
      or (DATASET_DIR / "sparse" / "0" / "cameras.txt").exists())

## 5. Phrase list — seed (optional), then curate

D15: an offline captioner **seeds** the phrase list and a human **curates** it.
Run **5a** to seed; then edit `PHRASES` in **5b**. If you'd rather skip the
captioner, just set `PHRASES` by hand and run 5b only.


In [ ]:
# 5a — seed phrases from frame captions (optional)
from worldsplat_prune.frontend.caption import HFCaptioner, seed_phrase_list

captioner = HFCaptioner("Salesforce/blip-image-captioning-base", device="cuda")
seed = seed_phrase_list(DATASET_DIR, captioner, n=12, out="phrases_seed.txt")
print(f"{len(seed)} seed phrases -> curate them in 5b")
seed[:40]

In [ ]:
# 5b — curated phrase list (edit me). Paste/trim the seed above or type your own.
PHRASES = [
    "wall", "floor", "ceiling", "table", "desk", "chair", "sofa",
    "cabinet", "shelf", "monitor", "door", "window", "whiteboard", "lamp",
]
print(len(PHRASES), "phrases")
PHRASES

## 6. GroundingDINO — phrases → boxes

Open-vocabulary detection: each phrase becomes zero or more boxes per frame. `STRIDE`
subsamples frames for the smoke test (set `1` for the full scene).


In [ ]:
from worldsplat_prune.frontend.grounding import detect_dataset

STRIDE = 20        # label every 20th frame for the test; 1 = full scene
RESIZE = 1024      # cap inference resolution to save memory; None = native

detections = detect_dataset(
    DATASET_DIR, PHRASES, "detections.json",
    device="cuda", stride=STRIDE, resize_long_side=RESIZE,
    box_threshold=0.30, text_threshold=0.25,
)

import collections
counts = collections.Counter()
for det in detections.values():
    for ph, boxes in det.items():
        counts[ph] += len(boxes)
print(f"{len(detections)} frames processed")
for ph, c in counts.most_common():
    print(f"  {ph:16s} {c}")

## 7. SAM2 box-prompt — boxes → per-class instance maps

Downloads SAM2.1-large once, then turns each box into a mask and writes one
instance map **per class** at native resolution: `semantic_instance_<slug>/<stem>.png`.
`stride=1` here (we only label the frames that have detections).


In [ ]:
from pathlib import Path
from worldsplat_prune.frontend.sam2_box import label_dataset_boxes

CKPT_DIR = Path(HOME_DIR) / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAM2_CKPT = CKPT_DIR / "sam2.1_hiera_large.pt"
if not SAM2_CKPT.exists():
    !wget -q --show-progress -O "{SAM2_CKPT}" https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

class_dirs = label_dataset_boxes(
    DATASET_DIR, detections,
    sam2_ckpt=str(SAM2_CKPT), device="cuda",
    min_area=100, max_area_frac=0.9, fill_void=True,
    resize_long_side=RESIZE, stride=1, overwrite=True,
)
class_dirs

## 8. Per-class fusion → composite instance-label map

Fuses each class independently (association never crosses classes) and composes the
canonical composite id `class_id * 10000 + instance_id` — the `L_v` the pruning
operator consumes. Class ids follow the curated `PHRASES` order.


In [ ]:
from worldsplat_prune.prep.fuse_instances import fuse_instances_per_class

class_ids = {p: i + 1 for i, p in enumerate(PHRASES)}
summary = fuse_instances_per_class(
    DATASET_DIR, class_dirs, class_ids=class_ids,
    metric="jaccard", min_score=0.25, min_views=2,
)
{ "composite_views": summary["composite_views"],
  "composite_points": summary["composite_points"],
  "class_ids": summary["class_ids"] }

## 9. Inspect the composite maps


In [ ]:
from pathlib import Path
from worldsplat_prune.datasets import get_adapter
from worldsplat_prune.datasets.composite import read_composite_map, decode, COMPOSITE_STRIDE

adapter = get_adapter("composite", DATASET_DIR)
print(adapter.meta())

comp_files = sorted((Path(DATASET_DIR) / "semantic_composite").glob("*.npy"))
print(len(comp_files), "composite maps")

if comp_files:
    ids = read_composite_map(comp_files[0])
    cls, inst = decode(ids)
    print("labeled px:", int((ids > 0).sum()),
          "| classes:", sorted(set(cls[cls > 0].tolist())),
          "| instances:", len({int(i) for i in inst[inst > 0].tolist()}))

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

if comp_files:
    ids = read_composite_map(comp_files[0])
    cls, inst = decode(ids)
    stem = comp_files[0].stem
    img_path = next(p for p in (Path(DATASET_DIR) / "images").iterdir() if p.stem == stem)
    rgb = np.array(Image.open(img_path).convert("RGB"))
    if cls.shape != rgb.shape[:2]:       # map may be at COLMAP camera resolution
        cls = cv2.resize(cls, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
        inst = cv2.resize(inst, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)

    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    ax[0].imshow(rgb); ax[0].set_title(f"RGB — {stem}")
    ax[1].imshow(np.where(cls > 0, cls, np.nan), cmap="tab20", interpolation="nearest")
    ax[1].set_title("class id")
    ax[2].imshow(np.where(inst > 0, inst, np.nan), cmap="nipy_spectral", interpolation="nearest")
    ax[2].set_title("instance id")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

## 10. (Optional) one-call driver + next steps

Once the stages above work, the whole front-end is a single call:
`prep.build_semantic_maps.build_semantic_maps(DATASET_DIR, PHRASES, sam2_ckpt=...)`
(runs detect → box-prompt → fuse → compose; `--skip_detect` reuses
`detections.json`, `--detect_only` stops before SAM2).

Then:

1. **Validate the classes** against Replica GT before trusting any pruning result:
   `python -m worldsplat_prune.prep.eval_instances --dataset_dir <scene> --class_gt semantic_class --level class --method frontend=semantic_composite`
2. **Train / eval** with `--label_adapter composite` (or `grounded_sam2`).
3. The pruning operator (§4–5), which consumes these maps, is the next component.

### Troubleshooting

- **CUDA OOM** → lower `RESIZE` (e.g. 768), use `sam2.1_hiera_small`, and/or raise
  `STRIDE`. SAM2 caches one image embedding per frame and is released after each.
- **No catalog `sam2`** → the `pip install git+.../sam2.git` line must succeed
  before cell 7 runs.
- **Empty composite dir** → the detector returned no boxes, or `min_views`/`min_area`
  filtered everything. Check the per-phrase counts (cell 6) and the `[fuse]` logs.
- **Stuff (wall/floor/ceiling) is sparse** → expected (risk §8.8): GroundingDINO
  detects things, not stuff. Prompt stuff explicitly or accept sparse labels.


In [ ]:
# from worldsplat_prune.prep.build_semantic_maps import build_semantic_maps
# build_semantic_maps(DATASET_DIR, PHRASES, sam2_ckpt=str(SAM2_CKPT), stride=1)